# 階段 5：第二測試年 2024（Colab Enterprise）

把 2024 當測試年（2021–2023 訓練），重跑所有需要 GPU 的模型，檢查 2025 的結論是否成立。
Persistence 與 XGBoost 系列已在本機跑完。微調版改用 2021–2022 訓練、2023 選學習率。
結果放在 `results/predictions_2024/`、`results/finetuned_2024/`，不會覆蓋 2025 的結果。

1. 連線到 **L4 GPU** 執行階段，把新的 `pm25_project.zip` 上傳到左側 📁 檔案瀏覽器（覆蓋舊檔）。
2. 按「全部執行」。這本最久，預估 3–5 小時（微調 3 變體 × 2 學習率，加上 15 個模型的 2024 推論）。
3. **中途斷線可以續跑**：重新「全部執行」即可，已完成的微調與預測會自動跳過。
4. 任何一格出錯都會停下，請貼 `Traceback` 開始到最後一行給 Claude。
5. 最後一格會下載 `pm25_2024_results.zip`（只有預測檔與摘要，不含模型權重）。
6. **跑完務必停止執行階段**。


In [ ]:
%%bash
set -e
# 1. 檢查 GPU 與 Python 版本
nvidia-smi || echo '沒有 GPU，將使用 CPU（較慢但可跑）'
python --version

In [ ]:
# 2. 解壓縮專案（自動尋找上傳的 zip；Colab Enterprise 的檔案瀏覽器根目錄是 /）
import os
candidates = ['/pm25_project.zip', '/content/pm25_project.zip',
              os.path.join(os.getcwd(), 'pm25_project.zip')]
found = [p for p in candidates if os.path.exists(p)]
assert found, '找不到 pm25_project.zip，請先上傳到左側檔案瀏覽器'
os.chdir(os.path.dirname(found[0]))
!rm -rf pm25/src pm25/data/processed
!unzip -q -o pm25_project.zip -d pm25
os.chdir('pm25')
print(os.getcwd())
!ls data/processed | head -3

In [ ]:
%%bash
set -e
# 3. 安裝套件（每次上傳新 zip 後都要重跑；numba 的 numpy 版本警告可忽略）
echo "目前目錄：$(pwd)"
test -f pyproject.toml || { echo "目錄不對：請先重跑第 2 格"; exit 1; }
pip uninstall -q -y pm25-tsfm || true
pip install -q '.[models]'
pip install -q --force-reinstall --no-deps .
cd / && python -c "import pandas, torch; from pm25.models.device import default_device; print('pandas', pandas.__version__, '| torch', torch.__version__, '| device =', default_device())"

## 零樣本與訓練模型（2024）

In [ ]:
%%bash
set -e
# 4. LSTM（L4 預估 10–20 分鐘）
# 執行階段重啟後工作目錄會跑掉：先切到解壓縮後的專案資料夾並確認資料存在
for d in "$(pwd)" /pm25 /content/pm25; do
  if [ -f "$d/pyproject.toml" ] && ls "$d"/data/processed/*.csv.gz >/dev/null 2>&1; then cd "$d"; break; fi
done
test -f data/processed/古亭.csv.gz || { echo "找不到資料：請重跑第 2 格"; exit 1; }
python -m pm25.experiment --family lstm --test-year 2024


In [ ]:
%%bash
set -e
# 5. Chronos-Bolt S/B 與 Chronos-2 uni／cov／oracle（L4 預估 30–60 分鐘）
# 執行階段重啟後工作目錄會跑掉：先切到解壓縮後的專案資料夾並確認資料存在
for d in "$(pwd)" /pm25 /content/pm25; do
  if [ -f "$d/pyproject.toml" ] && ls "$d"/data/processed/*.csv.gz >/dev/null 2>&1; then cd "$d"; break; fi
done
test -f data/processed/古亭.csv.gz || { echo "找不到資料：請重跑第 2 格"; exit 1; }
python -m pm25.experiment --family chronos --test-year 2024


In [ ]:
%%bash
set -e
# 6. LSTM 與 Chronos-2 加 JMA GSM 預報風場（L4 預估 20–40 分鐘）
# 執行階段重啟後工作目錄會跑掉：先切到解壓縮後的專案資料夾並確認資料存在
for d in "$(pwd)" /pm25 /content/pm25; do
  if [ -f "$d/pyproject.toml" ] && ls "$d"/data/processed/*.csv.gz >/dev/null 2>&1; then cd "$d"; break; fi
done
test -f data/processed/古亭.csv.gz || { echo "找不到資料：請重跑第 2 格"; exit 1; }
python -m pm25.experiment --family nwp_torch --test-year 2024


In [ ]:
%%bash
set -e
# 7. Chronos-2 四個鄰站變體（L4 預估 40–80 分鐘）
# 執行階段重啟後工作目錄會跑掉：先切到解壓縮後的專案資料夾並確認資料存在
for d in "$(pwd)" /pm25 /content/pm25; do
  if [ -f "$d/pyproject.toml" ] && ls "$d"/data/processed/*.csv.gz >/dev/null 2>&1; then cd "$d"; break; fi
done
test -f data/processed/古亭.csv.gz || { echo "找不到資料：請重跑第 2 格"; exit 1; }
python -m pm25.experiment --family spatial_chronos --test-year 2024


## 微調（2021–2022 訓練、2023 驗證）

In [ ]:
%%bash
set -e
# 8. 三個變體 × 兩個學習率的全參數微調
# 執行階段重啟後工作目錄會跑掉：先切到解壓縮後的專案資料夾並確認資料存在
for d in "$(pwd)" /pm25 /content/pm25; do
  if [ -f "$d/pyproject.toml" ] && ls "$d"/data/processed/*.csv.gz >/dev/null 2>&1; then cd "$d"; break; fi
done
test -f data/processed/古亭.csv.gz || { echo "找不到資料：請重跑第 2 格"; exit 1; }
python -m pm25.models.finetune --test-year 2024
cat results/finetuned_2024/*.json


In [ ]:
%%bash
set -e
# 9. 用微調後的模型預測 2024 年每小時起點
# 執行階段重啟後工作目錄會跑掉：先切到解壓縮後的專案資料夾並確認資料存在
for d in "$(pwd)" /pm25 /content/pm25; do
  if [ -f "$d/pyproject.toml" ] && ls "$d"/data/processed/*.csv.gz >/dev/null 2>&1; then cd "$d"; break; fi
done
test -f data/processed/古亭.csv.gz || { echo "找不到資料：請重跑第 2 格"; exit 1; }
python -m pm25.experiment --family finetuned --test-year 2024


## 打包與下載

In [ ]:
# 10. 確認 180 個預測檔（12 站 × 15 模型）與 3 個微調摘要後打包下載
import glob, os
for d in [os.getcwd(), '/pm25', '/content/pm25']:  # 執行階段重啟後工作目錄會跑掉
    if os.path.exists(f'{d}/data/processed/古亭.csv.gz'):
        os.chdir(d); break
print('專案目錄：', os.getcwd())
models = ['lstm', 'chronos_bolt_small', 'chronos_bolt_base', 'chronos2_uni', 'chronos2_cov',
          'chronos2_wind_oracle', 'lstm_nwp', 'chronos2_wind_nwp', 'chronos2_spatial',
          'chronos2_spatial_region', 'chronos2_spatial_all', 'chronos2_spatial_nwp',
          'chronos2_ft_uni', 'chronos2_ft_cov', 'chronos2_ft_nwp']
files = [f for m in models for f in glob.glob(f'results/predictions_2024/*__{m}.csv.gz')]
missing = [m for m in models if len(glob.glob(f'results/predictions_2024/*__{m}.csv.gz')) != 12]
files += glob.glob('results/finetuned_2024/*.json')
print('檔案數量：', len(files), '/ 183', '| 不齊全的模型：', missing)
assert len(files) == 183, '檔案不齊全，請往上檢查哪一格出錯'
!rm -f ../pm25_2024_results.zip
!zip -q ../pm25_2024_results.zip {' '.join(files)}
!ls -lh ../pm25_2024_results.zip
from google.colab import files as colab_files
colab_files.download('../pm25_2024_results.zip')


## 結束
到 Colab Enterprise →「執行階段」把這個執行階段**停止或刪除**，避免繼續計費。